# 01 · Data: CLICS⁴, Glottolog and the working concept set

Thin narrative notebook. All logic lives in `src/data/`; this notebook only reads what `python run.py data` wrote to `data/processed/` and `results/`. Decisions referenced as D-xx are in `DECISIONS.md`.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import config
from src.utils.io import read_parquet
from src.utils.concept_order import load_concept_order
load = lambda name: json.loads((config.RESULTS_DIR / name).read_text())

## 1. What was loaded (D-01, D-03)
Columns are resolved by CLDF property URL; the resolved schema is stored in `results/clics_schema.json`.

In [ ]:
schema = load('clics_schema.json')
{k: v for k, v in schema.items() if k != 'schema'}

In [ ]:
pd.DataFrame({comp: spec['columns'] for comp, spec in schema['schema'].items()}).fillna('')

## 2. Languages and families (D-02, D-04, D-06)
Varieties collapse to Glottolog language-level codes; isolates are one-language families; pseudo-families are excluded.

In [ ]:
langs = load('language_summary.json')
{k: v for k, v in langs.items() if k != 'languages_per_family_top15'}

In [ ]:
pd.Series(langs['languages_per_family_top15'], name='languages').to_frame()

Note for H3: Indo-European contributes only 63 languages, against 401 Austronesian. Data density in CLICS and data density in model training run in opposite directions, which is why H3 uses rarefaction (D-25).

## 3. Concept threshold (D-05)
The knee of the coverage curve, constrained to 300–500 concepts.

In [ ]:
from IPython.display import Image
Image(filename=str(config.FIGURES_DIR / 'figS1_concept_coverage.png'), width=760)

In [ ]:
thr = load('concept_threshold.json')
{k: thr[k] for k in ['knee_threshold','knee_n_concepts','threshold_used','robustness_threshold','anchors_added_below_threshold','n_excluded_by_override','n_gloss_overrides','n_working','n_robustness_set']}

In [ ]:
pd.DataFrame(thr['knee_sensitivity_to_candidate_range'])

## 4. The working set (D-11, D-12, D-35)

In [ ]:
w = pd.read_csv(config.RESULTS_DIR / 'working_concepts.csv')
order = load_concept_order()
assert w['concept_id'].tolist() == order
w['semantic_field'].value_counts().to_frame('concepts')

In [ ]:
w[w['gloss_source'] == 'override'][['concept_id', 'gloss', 'collision_group']]

## 5. The *kal* case in CLICS⁴
Hindi (NorthEuraLex) gives one form for YESTERDAY and TOMORROW; the Urdu list in CLICS⁴ has neither concept.

In [ ]:
forms = read_parquet(config.PROCESSED_DIR / 'forms.parquet')
languages = read_parquet(config.PROCESSED_DIR / 'languages.parquet')
hu = languages[languages['iso639_3'].isin(['hin', 'urd'])][['variety_id', 'variety_name', 'contribution_id']]
forms.merge(hu, on='variety_id').query("concept_id in ['YESTERDAY', 'TOMORROW']")

## 6. English colexification candidates (D-24, D-34)
High-recall WordNet candidates; the audit sheet is generated once the network exists (Phase 2) and audited in Phase 6.

In [ ]:
ec = load('english_colex_summary.json')
{k: v for k, v in ec.items() if k != 'audit_criteria'}

In [ ]:
cands = read_parquet(config.PROCESSED_DIR / 'english_colex_candidates.parquet')
key = [('ARM','HAND'),('TREE','WOOD'),('TOMORROW','YESTERDAY'),('FINGER','TOE'),('SEE','UNDERSTAND'),('LANGUAGE','TONGUE')]
pd.DataFrame([{'pair': f'{a}~{b}', 'candidate': ((cands.concept_a==a)&(cands.concept_b==b)).any(), 'route': cands.loc[(cands.concept_a==a)&(cands.concept_b==b), 'evidence_route'].squeeze() if ((cands.concept_a==a)&(cands.concept_b==b)).any() else ''} for a, b in key])